
# 🧠 Sieć neuronowa do klasyfikacji — 5 wariantów rozwiązania

## Cel notebooka
Ten notebook pokazuje **pięć wariantów rozwiązania tego samego problemu klasyfikacyjnego**.

Nie chodzi o znalezienie jednej „magicznej” architektury.  
Chodzi o pokazanie kursantom, jak kolejne decyzje wpływają na:

- szybkość uczenia,
- stabilność treningu,
- przeuczenie,
- jakość walidacji,
- jakość na zbiorze testowym,
- precision / recall / F1,
- ROC-AUC.

## Pięć wariantów

### Wariant 1 — Baseline
Prosta sieć, wysoki learning rate, duży batch, brak regularyzacji.

### Wariant 2 — Mniejszy learning rate + więcej epok
Sprawdzamy, czy spokojniejszy trening poprawi stabilność.

### Wariant 3 — Większa sieć + Dropout
Sprawdzamy, czy większa pojemność modelu pomoże, gdy jednocześnie ograniczymy przeuczenie.

### Wariant 4 — BatchNormalization + EarlyStopping
Dodajemy mechanizmy stabilizujące i automatyczne zatrzymanie treningu.

### Wariant 5 — Model końcowy + optymalizacja progu
Łączymy kilka dobrych praktyk i pokazujemy, że **próg 0.5 nie zawsze jest najlepszy**.

> Dataset: Breast Cancer Wisconsin z `scikit-learn`.  
> Notebook ma charakter wyłącznie edukacyjny.


## 1. Import bibliotek

In [ ]:

import os
import sys
import random
import subprocess
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve
)

if importlib.util.find_spec("tensorflow") is None:
    print("TensorFlow nie jest zainstalowany. Instaluję...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)


## 2. Dane i przygotowanie zbiorów

In [ ]:

data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Dane:", X.shape)
print("Klasy:", dict(enumerate(data.target_names)))
display(X.head())


In [ ]:

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=SEED,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)



## 3. Funkcje pomocnicze

Będziemy używać tych samych funkcji do oceny każdego wariantu, aby porównanie było uczciwe.


In [ ]:

def plot_history(history, title):
    hist = pd.DataFrame(history.history)
    hist["epoch"] = np.arange(1, len(hist) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(hist["epoch"], hist["loss"], label="train loss")
    plt.plot(hist["epoch"], hist["val_loss"], label="validation loss")
    plt.title(f"{title} — loss")
    plt.xlabel("Epoka")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(hist["epoch"], hist["accuracy"], label="train accuracy")
    plt.plot(hist["epoch"], hist["val_accuracy"], label="validation accuracy")
    plt.title(f"{title} — accuracy")
    plt.xlabel("Epoka")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()


def evaluate_model(model, X_test_data, y_test_data, threshold=0.5, name="model"):
    probs = model.predict(X_test_data, verbose=0).ravel()
    preds = (probs >= threshold).astype(int)

    metrics = {
        "model": name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_test_data, preds),
        "precision": precision_score(y_test_data, preds),
        "recall": recall_score(y_test_data, preds),
        "f1": f1_score(y_test_data, preds),
        "roc_auc": roc_auc_score(y_test_data, probs)
    }

    print(pd.DataFrame([metrics]).round(4).to_string(index=False))
    print("\nClassification report:")
    print(classification_report(
        y_test_data,
        preds,
        target_names=data.target_names,
        digits=4
    ))

    cm = confusion_matrix(y_test_data, preds)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=data.target_names
    )
    disp.plot(values_format="d")
    plt.title(f"Macierz pomyłek — {name}")
    plt.show()

    fpr, tpr, _ = roc_curve(y_test_data, probs)
    auc = roc_auc_score(y_test_data, probs)

    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC — {name}")
    plt.legend()
    plt.show()

    return metrics, probs, preds


results = []



# WARIANT 1 — Baseline

## Idea
Zaczynamy od prostej sieci:
- 2 warstwy ukryte,
- `ReLU`,
- brak Dropoutu,
- brak BatchNormalization,
- brak EarlyStopping,
- dość wysoki `learning_rate`,
- duży `batch_size`.

## Co pokazujemy kursantom?
To jest **punkt odniesienia**. Model działa, ale nie wykorzystujemy żadnych mechanizmów stabilizujących ani regularyzujących.

Warto zwrócić uwagę, czy:
- loss walidacyjny zaczyna rosnąć,
- accuracy treningowe odjeżdża od walidacyjnego,
- wyniki są stabilne.


In [ ]:

def build_model_v1(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dense(8, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.01),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc")
        ]
    )
    return model

model_v1 = build_model_v1(X_train_scaled.shape[1])

history_v1 = model_v1.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=25,
    batch_size=128,
    verbose=1
)

plot_history(history_v1, "Wariant 1")
metrics_v1, probs_v1, preds_v1 = evaluate_model(
    model_v1, X_test_scaled, y_test,
    threshold=0.5,
    name="Wariant 1 — baseline"
)
results.append(metrics_v1)



# WARIANT 2 — Mniejszy learning rate + więcej epok

## Zmiana
Zmniejszamy:
- `learning_rate` z `0.01` do `0.001`.

Zwiększamy:
- liczbę epok,
- zmniejszamy `batch_size`.

## Dlaczego?
Za duży learning rate może powodować, że optymalizator „przeskakuje” dobre minima funkcji kosztu.

Mniejszy learning rate oznacza zwykle:
- wolniejsze,
- bardziej stabilne,
- bardziej precyzyjne uczenie.

Mniejszy batch daje więcej aktualizacji wag w trakcie jednej epoki.

## Co obserwować?
- czy loss spada płynniej,
- czy walidacja staje się stabilniejsza,
- czy więcej epok faktycznie pomaga.


In [ ]:

def build_model_v2(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dense(8, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc")
        ]
    )
    return model

model_v2 = build_model_v2(X_train_scaled.shape[1])

history_v2 = model_v2.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=60,
    batch_size=32,
    verbose=1
)

plot_history(history_v2, "Wariant 2")
metrics_v2, probs_v2, preds_v2 = evaluate_model(
    model_v2, X_test_scaled, y_test,
    threshold=0.5,
    name="Wariant 2 — spokojniejszy trening"
)
results.append(metrics_v2)



# WARIANT 3 — Większa sieć + Dropout

## Zmiana
Zwiększamy pojemność modelu:
- 64 neurony,
- 32 neurony,
- 16 neuronów.

Dodajemy:
- `Dropout(0.3)`.

## Dlaczego?
Większa sieć może nauczyć się bardziej złożonych zależności, ale rośnie ryzyko przeuczenia.

Dropout podczas treningu losowo wyłącza część neuronów.  
Dzięki temu sieć trudniej „zapamiętuje” dane treningowe i musi nauczyć się bardziej odpornych reprezentacji.

## Co obserwować?
- czy train accuracy spada trochę względem bardzo dużej sieci bez Dropoutu,
- czy validation accuracy / AUC rośnie,
- czy różnica między train i validation maleje.


In [ ]:

def build_model_v3(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.30),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.30),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc")
        ]
    )
    return model

model_v3 = build_model_v3(X_train_scaled.shape[1])

history_v3 = model_v3.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=70,
    batch_size=32,
    verbose=1
)

plot_history(history_v3, "Wariant 3")
metrics_v3, probs_v3, preds_v3 = evaluate_model(
    model_v3, X_test_scaled, y_test,
    threshold=0.5,
    name="Wariant 3 — Dropout"
)
results.append(metrics_v3)



# WARIANT 4 — BatchNormalization + EarlyStopping + ReduceLROnPlateau

## Zmiana
Dodajemy trzy mechanizmy:

### 1. BatchNormalization
Normalizuje aktywacje pomiędzy warstwami i często stabilizuje uczenie.

### 2. EarlyStopping
Kończy trening, kiedy wynik walidacyjny przestaje się poprawiać.

### 3. ReduceLROnPlateau
Zmniejsza learning rate, gdy model przestaje robić postęp.

## Dlaczego?
To wariant pokazujący, że hiperparametrem jest również **sposób prowadzenia treningu**, nie tylko architektura.

## Co obserwować?
- ile epok rzeczywiście zostanie wykonanych,
- kiedy learning rate zostanie zmniejszony,
- czy model kończy uczenie zanim zacznie się przeuczać.


In [ ]:

def build_model_v4(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),

        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.25),

        layers.Dense(32, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.20),

        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc")
        ]
    )
    return model

model_v4 = build_model_v4(X_train_scaled.shape[1])

callbacks_v4 = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=12,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

history_v4 = model_v4.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=150,
    batch_size=32,
    callbacks=callbacks_v4,
    verbose=1
)

plot_history(history_v4, "Wariant 4")
metrics_v4, probs_v4, preds_v4 = evaluate_model(
    model_v4, X_test_scaled, y_test,
    threshold=0.5,
    name="Wariant 4 — callbacks"
)
results.append(metrics_v4)



# WARIANT 5 — Regularyzacja L2 + callbacks + dobór progu

## Zmiana
W tym wariancie łączymy:
- umiarkowanie dużą architekturę,
- `BatchNormalization`,
- `Dropout`,
- regularyzację `L2`,
- `EarlyStopping`,
- `ReduceLROnPlateau`.

Następnie **nie przyjmujemy automatycznie progu 0.5**.

## Dlaczego próg jest ważny?
Sieć zwraca prawdopodobieństwo.  
Dopiero próg decyduje, od jakiego prawdopodobieństwa przypiszemy klasę 1.

Przykład:
- próg 0.5 → klasyczny wybór,
- próg 0.3 → więcej przypadków zaklasyfikujemy jako klasę 1,
- próg 0.7 → model będzie bardziej ostrożny.

Dobór progu może silnie zmienić:
- precision,
- recall,
- F1.

## Cel
Znajdziemy próg, który maksymalizuje **F1 na zbiorze walidacyjnym**, a dopiero potem zastosujemy go do testu.


In [ ]:

def build_model_v5(input_dim):
    l2 = regularizers.l2(0.001)

    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),

        layers.Dense(64, activation="relu", kernel_regularizer=l2),
        layers.BatchNormalization(),
        layers.Dropout(0.25),

        layers.Dense(32, activation="relu", kernel_regularizer=l2),
        layers.BatchNormalization(),
        layers.Dropout(0.20),

        layers.Dense(16, activation="relu", kernel_regularizer=l2),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc")
        ]
    )
    return model

model_v5 = build_model_v5(X_train_scaled.shape[1])

callbacks_v5 = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

history_v5 = model_v5.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=200,
    batch_size=32,
    callbacks=callbacks_v5,
    verbose=1
)

plot_history(history_v5, "Wariant 5")



## Dobór najlepszego progu na walidacji

**Bardzo ważne:** progu nie dobieramy na teście.

Test powinien być użyty dopiero po zakończeniu wszystkich decyzji projektowych.


In [ ]:

val_probs_v5 = model_v5.predict(X_val_scaled, verbose=0).ravel()

thresholds = np.arange(0.10, 0.91, 0.01)
threshold_results = []

for threshold in thresholds:
    val_preds = (val_probs_v5 >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_val, val_preds, zero_division=0),
        "recall": recall_score(y_val, val_preds, zero_division=0),
        "f1": f1_score(y_val, val_preds, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results)

best_row = threshold_df.loc[threshold_df["f1"].idxmax()]
best_threshold = float(best_row["threshold"])

display(threshold_df.sort_values("f1", ascending=False).head(10).round(4))

print(f"Najlepszy próg wg F1 na walidacji: {best_threshold:.2f}")
print(f"Walidacyjne F1: {best_row['f1']:.4f}")


In [ ]:

plt.figure(figsize=(8, 5))
plt.plot(threshold_df["threshold"], threshold_df["precision"], label="precision")
plt.plot(threshold_df["threshold"], threshold_df["recall"], label="recall")
plt.plot(threshold_df["threshold"], threshold_df["f1"], label="F1")
plt.axvline(best_threshold, linestyle="--", label=f"best threshold = {best_threshold:.2f}")
plt.xlabel("Próg klasyfikacji")
plt.ylabel("Wartość metryki")
plt.title("Wpływ progu na metryki — walidacja")
plt.legend()
plt.show()


In [ ]:

metrics_v5, probs_v5, preds_v5 = evaluate_model(
    model_v5,
    X_test_scaled,
    y_test,
    threshold=best_threshold,
    name="Wariant 5 — model końcowy + tuning progu"
)
results.append(metrics_v5)



# 6. Porównanie wszystkich wariantów

Tutaj widać, że „lepszy model” nie oznacza automatycznie:
- największej sieci,
- największej accuracy,
- największej liczby epok.

Patrzymy na cały zestaw metryk.


In [ ]:

results_df = pd.DataFrame(results)
results_df = results_df[
    ["model", "threshold", "accuracy", "precision", "recall", "f1", "roc_auc"]
]

display(results_df.round(4).sort_values("f1", ascending=False))


In [ ]:

metrics_only = results_df.set_index("model")[["accuracy", "precision", "recall", "f1", "roc_auc"]]

metrics_only.plot(kind="bar", figsize=(12, 6))
plt.title("Porównanie pięciu wariantów")
plt.ylabel("Wartość metryki")
plt.xticks(rotation=25, ha="right")
plt.ylim(0.80, 1.02)
plt.legend(loc="lower right")
plt.show()



# 7. Jak omawiać te warianty na szkoleniu?

## Wariant 1
**Pytanie:** Czy działający model oznacza dobry model?

Pokazuje, że nawet prosta sieć może osiągać wysokie wyniki, ale nie oznacza to, że proces treningu został dobrze zaprojektowany.

---

## Wariant 2
**Pytanie:** Co robi learning rate?

To bardzo dobry moment, żeby pokazać analogię:

> Learning rate to długość kroku podczas schodzenia z góry we mgle.

Za duży krok:
- szybko schodzimy,
- ale możemy przeskoczyć minimum.

Za mały:
- ruch jest precyzyjny,
- ale trwa długo.

---

## Wariant 3
**Pytanie:** Czy większa sieć zawsze jest lepsza?

Nie.

Więcej neuronów zwiększa zdolność modelu do dopasowania danych, ale także zwiększa ryzyko przeuczenia.

Dropout działa jak wymuszanie pracy zespołowej: neuron nie może zakładać, że jego „koledzy” zawsze będą obecni.

---

## Wariant 4
**Pytanie:** Czy liczba epok powinna być ustawiona ręcznie?

Nie zawsze.

Możemy ustawić np. 150 epok jako maksimum, ale pozwolić `EarlyStopping` zdecydować, kiedy dalsze uczenie nie ma już sensu.

`ReduceLROnPlateau` działa jak redukcja kroku, gdy optymalizator zbliża się do dobrego minimum.

---

## Wariant 5
**Pytanie:** Dlaczego 0.5?

To jedno z najważniejszych pytań.

W wielu projektach próg `0.5` jest przyjmowany mechanicznie, mimo że wymagania biznesowe mogą preferować:
- recall,
- precision,
- F1,
- minimalizację false negatives,
- minimalizację false positives.

W prawdziwych projektach **próg decyzji jest hiperparametrem systemu**, nawet jeśli nie jest hiperparametrem samej sieci.



# 8. Ćwiczenie dla kursantów

Na podstawie pięciu pokazanych rozwiązań zbuduj **wariant 6**.

Możesz eksperymentować z:

- `ELU`, `tanh`, `LeakyReLU`,
- `SGD`,
- `RMSprop`,
- innym learning rate,
- innym batch size,
- większą / mniejszą siecią,
- Dropout 0.1–0.5,
- inną siłą L2,
- innym patience,
- innym kryterium EarlyStopping,
- optymalizacją progu pod recall albo precision.

## Warunek
Twój wariant musi zawierać:
1. hipotezę,
2. zmianę,
3. wynik,
4. porównanie z baseline,
5. wniosek.

### Przykład hipotezy
> „Zmniejszenie learning rate z 0.001 do 0.0003 i zastosowanie większej liczby epok poprawi stabilność walidacji i ROC-AUC.”

Po treningu sprawdź, czy hipoteza była prawdziwa.


In [ ]:

# ============================================
# WARIANT 6 — MIEJSCE DLA KURSANTA
# ============================================

def build_model_v6(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),

        # TODO: zaprojektuj własną architekturę

        layers.Dense(32, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc")
        ]
    )

    return model

# model_v6 = build_model_v6(X_train_scaled.shape[1])
# ...
